In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from mapie.regression import SplitConformalRegressor
from sklearn.linear_model import ElasticNet
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

PROC_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results")

expr = pd.read_csv(PROC_DIR / "expression_filtered.csv", index_col='cbio_patient_id')
drug = pd.read_csv(PROC_DIR / "drug_final.csv")

with open(RESULTS_DIR / "models/trained_models.pkl", "rb") as f:
    models = pickle.load(f)

print("Expression:", expr.shape)
print("expr index sample:", expr.index[:3].tolist())
print("Drug rows:", len(drug))
print("Models loaded:", len(models))

Expression: (318, 14121)
expr index sample: ['aml_ohsu_2022_2003', 'aml_ohsu_2022_2005', 'aml_ohsu_2022_2008']
Drug rows: 34764
Models loaded: 122


In [4]:
# define split function and test on Sorafenib

def make_splits(drug_name, drug_df, expr_df, random_state=42):
    subset = drug_df[drug_df['inhibitor'] == drug_name][['cbio_patient_id', 'auc']].drop_duplicates()
    subset = subset[subset['cbio_patient_id'].isin(expr_df.index)]
    X = expr_df.loc[subset['cbio_patient_id']].values
    y = subset['auc'].values
    n = len(y)
    if n < 30:
        return None
    idx = np.random.RandomState(random_state).permutation(n)
    train_end = int(n * 0.6)
    calib_end = int(n * 0.8)
    return {
        'X_train': X[idx[:train_end]], 'y_train': y[idx[:train_end]],
        'X_calib': X[idx[train_end:calib_end]], 'y_calib': y[idx[train_end:calib_end]],
        'X_test': X[idx[calib_end:]], 'y_test': y[idx[calib_end:]]
    }

drug_name = "Sorafenib"
splits = make_splits(drug_name, drug, expr)
print(f"Train: {len(splits['y_train'])} | Calib: {len(splits['y_calib'])} | Test: {len(splits['y_test'])}")

en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
en.fit(splits['X_train'], splits['y_train'])

mapie_model = SplitConformalRegressor(en, prefit=True, confidence_level=0.9)
mapie_model.conformalize(splits['X_calib'], splits['y_calib'])

y_pred, intervals_raw = mapie_model.predict_interval(splits['X_test'])
lower = intervals_raw[:, 0, 0]
upper = intervals_raw[:, 1, 0]

coverage = np.mean((splits['y_test'] >= lower) & (splits['y_test'] <= upper))
widths = upper - lower

print(f"Target coverage: 90%")
print(f"Empirical coverage: {coverage:.4f}")
print(f"Mean interval width: {widths.mean():.4f}")

Train: 211 | Calib: 70 | Test: 71
Target coverage: 90%
Empirical coverage: 0.8732
Mean interval width: 139.0577


In [5]:
# run across all 122 drugs at 80%, 90%, 95%

confidence_levels = [0.80, 0.90, 0.95]
cp_results = []

for drug_name in drug['inhibitor'].unique():
    splits = make_splits(drug_name, drug, expr)
    if splits is None:
        continue

    en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
    en.fit(splits['X_train'], splits['y_train'])

    row = {'drug': drug_name, 'n_train': len(splits['y_train']),
           'n_calib': len(splits['y_calib']), 'n_test': len(splits['y_test'])}

    for cl in confidence_levels:
        min_samples = int(1 / (1 - cl)) + 1
        if len(splits['y_calib']) < min_samples:
            row[f'coverage_{int(cl*100)}'] = np.nan
            row[f'width_{int(cl*100)}'] = np.nan
            continue
        try:
            mapie_model = SplitConformalRegressor(en, prefit=True, confidence_level=cl)
            mapie_model.conformalize(splits['X_calib'], splits['y_calib'])
            y_pred, intervals_raw = mapie_model.predict_interval(splits['X_test'])
            lower = intervals_raw[:, 0, 0]
            upper = intervals_raw[:, 1, 0]
            coverage = np.mean((splits['y_test'] >= lower) & (splits['y_test'] <= upper))
            width = (upper - lower).mean()
            row[f'coverage_{int(cl*100)}'] = round(coverage, 4)
            row[f'width_{int(cl*100)}'] = round(width, 4)
        except Exception:
            row[f'coverage_{int(cl*100)}'] = np.nan
            row[f'width_{int(cl*100)}'] = np.nan

    cp_results.append(row)

cp_df = pd.DataFrame(cp_results)
cp_df.to_csv(RESULTS_DIR / "conformal_results.csv", index=False)
print(f"Completed: {len(cp_df)} drugs")
print("\nMean empirical coverage vs nominal:")
for cl in confidence_levels:
    col = f'coverage_{int(cl*100)}'
    print(f"  {int(cl*100)}%: {cp_df[col].mean():.4f} (target {cl})")
print("\nMean interval width:")
for cl in confidence_levels:
    col = f'width_{int(cl*100)}'
    print(f"  {int(cl*100)}%: {cp_df[col].mean():.4f}")

Completed: 122 drugs

Mean empirical coverage vs nominal:
  80%: 0.8141 (target 0.8)
  90%: 0.9074 (target 0.9)
  95%: 0.9550 (target 0.95)

Mean interval width:
  80%: 122.5917
  90%: 162.4115
  95%: 200.3193


In [6]:
# 

uncertainty_records = []

for drug_name in drug['inhibitor'].unique():
    splits = make_splits(drug_name, drug, expr)
    if splits is None:
        continue

    subset = drug[drug['inhibitor'] == drug_name][['cbio_patient_id', 'auc']].drop_duplicates()
    subset = subset[subset['cbio_patient_id'].isin(expr.index)]

    n = len(subset)
    idx = np.random.RandomState(42).permutation(n)
    calib_end = int(n * 0.8)
    test_patients = subset['cbio_patient_id'].values[idx[calib_end:]]

    en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
    en.fit(splits['X_train'], splits['y_train'])

    mapie_model = SplitConformalRegressor(en, prefit=True, confidence_level=0.9)
    mapie_model.conformalize(splits['X_calib'], splits['y_calib'])
    y_pred, intervals_raw = mapie_model.predict_interval(splits['X_test'])
    lower = intervals_raw[:, 0, 0]
    upper = intervals_raw[:, 1, 0]
    widths = upper - lower

    for i, patient in enumerate(test_patients):
        uncertainty_records.append({
            'drug': drug_name,
            'cbio_patient_id': patient,
            'y_pred': round(y_pred[i], 4),
            'lower': round(lower[i], 4),
            'upper': round(upper[i], 4),
            'width': round(widths[i], 4),
            'y_true': splits['y_test'][i]
        })

uncertainty_df = pd.DataFrame(uncertainty_records)
uncertainty_df.to_csv(RESULTS_DIR / "uncertainty_per_patient.csv", index=False)
print("Records saved:", len(uncertainty_df))
print("Unique patients:", uncertainty_df['cbio_patient_id'].nunique())
print("Unique drugs:", uncertainty_df['drug'].nunique())
print("\nWidth summary:")
print(uncertainty_df['width'].describe())

Records saved: 7000
Unique patients: 312
Unique drugs: 122

Width summary:
count    7000.000000
mean      158.401257
std        36.910353
min        77.526800
25%       131.905000
50%       154.189000
75%       185.751600
max       272.111400
Name: width, dtype: float64


In [7]:
cp_df = pd.read_csv(RESULTS_DIR / "conformal_results.csv")
print("Drugs with valid coverage_95:", cp_df['coverage_95'].notna().sum())
print("Drugs with valid coverage_90:", cp_df['coverage_90'].notna().sum())
print("Drugs with valid coverage_80:", cp_df['coverage_80'].notna().sum())

Drugs with valid coverage_95: 114
Drugs with valid coverage_90: 122
Drugs with valid coverage_80: 122
